# Company News Monitor

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/company-news-monitor.ipynb)

**What:** Company news monitoring at $0.002 per article from 75 publisher RSS feeds (Bloomberg, CNBC, FT, TechCrunch, PR Newswire...) plus the GDELT news index when reachable. Track companies, domains or keywords. Deduped, event-tagged, delta mode for schedules.

**Why:** run the [`company-news-monitor`](https://apify.com/transparent_meteorite/company-news-monitor) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/company-news-monitor) for every option.

In [2]:
run_input = {'companies': ['Nvidia', 'Stripe'], 'timespan': '24h', 'maxItems': 5}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/company-news-monitor').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.company-news-monitor runId:vfCCnIQMsFyoOctch] -> Status: RUNNING, Message: 


[apify.company-news-monitor runId:vfCCnIQMsFyoOctch] -> 2026-10-13T22:17:43.458Z ACTOR: Pulling container image of build reIBINSqHSWEXgAO8 from registry.


[apify.company-news-monitor runId:vfCCnIQMsFyoOctch] -> 2026-10-13T22:17:44.195Z ACTOR: Creating container.


[apify.company-news-monitor runId:vfCCnIQMsFyoOctch] -> 2026-10-13T22:17:44.263Z ACTOR: Starting container.


[apify.company-news-monitor runId:vfCCnIQMsFyoOctch] -> 2026-10-13T22:17:44.264Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.company-news-monitor runId:vfCCnIQMsFyoOctch] -> 2026-10-13T22:17:57.478Z Warnings: GDELT: HTTP 429 (RSS results still delivered)


[apify.company-news-monitor runId:vfCCnIQMsFyoOctch] -> 2026-10-13T22:17:57.612Z INFO  [Status message]: Done: 5 items


[apify.company-news-monitor runId:vfCCnIQMsFyoOctch] -> 2026-10-13T22:17:57.685Z Thanks for using Company News Monitor. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/ZcqPk0hMvnVkcwVfM/issues) with the run URL and we will reply there.


[apify.company-news-monitor runId:vfCCnIQMsFyoOctch] -> Status: SUCCEEDED, Message: Done: 5 items


Run status: SUCCEEDED | items: 5


## 4. Tabulate

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["title", "sourceName", "publishedAt", "country", "sentiment", "matchedKeywords"] if c in df.columns]
df[SHOW]

,title,sourceName,publishedAt,country,sentiment,matchedKeywords
0,Tinygrad refreshes Tinybox with an 'old' EPYC CPU running DDR4 RAM...,TechRadar,2026-10-13T20:30:00.000Z,GB,neutral,[Nvidia]
1,Nvidia Has an Employee Turnover Rate of Just 3.7 Percent. Its Lead...,Inc.,2026-10-13T19:30:00.000Z,US,neutral,[Nvidia]
2,What is NVIDIA DLSS and how does it work?,Engadget,2026-10-13T18:30:00.000Z,US,neutral,[Nvidia]
3,Nvidia CEO Announces $2 Billion in Europe Deals During Tour,Bloomberg Technology,2026-10-13T16:07:12.000Z,US,neutral,[Nvidia]
4,"On theCUBE Pod: AI doubters get loud, and IBM seeks competitive ad...",SiliconANGLE,2026-10-13T14:38:45.000Z,US,negative,[Nvidia]


## 5. A quick analysis

In [5]:
print(df['sentiment'].value_counts(dropna=False))
df['sourceName'].value_counts().head(10)

sentiment
neutral     4
negative    1
Name: count, dtype: int64


sourceName
TechRadar               1
Inc.                    1
Engadget                1
Bloomberg Technology    1
SiliconANGLE            1
Name: count, dtype: int64

## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [company-news-monitor on the Apify Store](https://apify.com/transparent_meteorite/company-news-monitor).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.